# Step 8c: spend trend and return rate
Two features, judged by **rule E** (decided in 8b) against the model we currently keep (R, F, M + interval + overdue ratio).

- **RecentShare** = share of a customer's gross purchase spend that falls in the last 90 days before the cutoff (0 = nothing recent, 1 = all recent).
- **ReturnRate** = return invoices / purchase invoices, counted inside the feature window only (the old project's mistake was using all history).

In [1]:
import sys, warnings; sys.path.insert(0, '../src'); warnings.filterwarnings('ignore')
import pandas as pd
from step8_features import *
pd.set_option('display.width', 200)
train = make_snapshot(load_ledger(), TRAIN_CUTOFF); y = train.target
base = ('IPI', 'OverdueRatio')
X_rfm, X_kept = build_features(train), build_features(train, base)
build_features(train, ('RecentShare', 'ReturnRate'))[['RecentShare', 'ReturnRate']].describe().round(3)

,RecentShare,ReturnRate
count,4242.000,4242.000
mean,0.410,0.199
std,0.385,0.358
min,0.000,0.000
25%,0.000,0.000
50%,0.338,0.000
75%,0.738,0.300
max,1.000,5.000


## Screen each candidate against the kept model (fold splits for information, customer bootstrap for the verdict)

In [2]:
results = {}
for extra in (('RecentShare',), ('ReturnRate',), ('RecentShare', 'ReturnRate')):
    X_new = build_features(train, base + extra)
    t = screen(X_kept, X_new, y); bt = customer_bootstrap(X_kept, X_new, y)
    results[extra] = (t, bt)
    print('=== +', ' + '.join(extra), '(on top of R,F,M + IPI + OverdueRatio)')
    print(summarize(t).round(4).to_string())
    print({k: tuple(round(x, 4) for x in v) for k, v in bt.items()}, '\n')

=== + RecentShare (on top of R,F,M + IPI + OverdueRatio)
                  mean      sd  improved       t
R2              0.0245  0.0125      10.0  6.2111
MAE gain (GBP)  4.2342  3.1418       9.0  4.2618
Capture@20%     0.0008  0.0016       6.0  1.6363
{'R2': (0.0099, -0.0148, 0.0296), 'MAE gain (GBP)': (1.5104, -5.6077, 9.9312), 'Capture@20%': (0.0013, -0.0031, 0.0055)} 



=== + ReturnRate (on top of R,F,M + IPI + OverdueRatio)
                  mean      sd  improved       t
R2             -0.0026  0.0091       5.0 -0.8904
MAE gain (GBP)  0.4448  1.3484       7.0  1.0432
Capture@20%    -0.0006  0.0008       4.0 -2.4047
{'R2': (0.0031, -0.0063, 0.0104), 'MAE gain (GBP)': (0.9385, -1.7429, 4.2511), 'Capture@20%': (-0.0004, -0.0029, 0.0018)} 



=== + RecentShare + ReturnRate (on top of R,F,M + IPI + OverdueRatio)
                  mean      sd  improved       t
R2              0.0243  0.0119      10.0  6.4320
MAE gain (GBP)  3.6068  2.9519       9.0  3.8639
Capture@20%     0.0006  0.0015       7.0  1.1468
{'R2': (0.0135, -0.0066, 0.0348), 'MAE gain (GBP)': (1.1969, -4.7652, 8.2234), 'Capture@20%': (0.0011, -0.0035, 0.0056)} 



In [3]:
rows = {}
for extra, (t, bt) in results.items():
    v = rule_verdicts(t, bt)
    rows[' + '.join(extra)] = {k.split(':')[0]: ('KEEP' if val else 'reject') for k, val in v.items()}
pd.DataFrame(rows).T

,A,B,C,D,E
RecentShare,KEEP,KEEP,KEEP,reject,reject
ReturnRate,reject,reject,reject,reject,reject
RecentShare + ReturnRate,KEEP,KEEP,KEEP,reject,reject


## What we learned
- **RecentShare is the clearest lesson so far.** Fold splits look great (R² better in 10/10, t = 6), so rules A to C would keep it. Over customers the R² interval is -0.015 to +0.030 and the MAE interval -5.6 to +9.9 GBP: the gain is not distinguishable from zero. Rule E rejects it.
- **ReturnRate** adds nothing (R² +0.003, MAE +0.9 GBP, both intervals straddle 0). On its own, on plain R, F, M, it makes the model significantly worse in MAE (-4.0).
- Together: also rejected.
- Current kept set is unchanged: R, F, M + IPI + OverdueRatio (provisional).